In [11]:
%pip install -q transformers datasets accelerate evaluate seqeval ftfy nltk pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
import re
import json
import glob
import torch
import torch.nn as nn
import nltk
import ftfy
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, AutoModelForTokenClassification, logging as tf_logging

tf_logging.set_verbosity_error()
tf_logging.disable_progress_bar()

from datasets import logging as ds_logging
ds_logging.set_verbosity_error()
ds_logging.disable_progress_bar()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cuda


In [13]:
model_choice = "electra-small"
print(f"Selected model: {model_choice}")


Selected model: electra-small


In [ ]:

fallback_map = {
    "electra-small": "google/electra-small-discriminator",
    "bert-tiny": "prajjwal1/bert-tiny",
    "tinybert": "huawei-noah/TinyBERT_General_4L_312D",
    "legal-bert": "nlpaueb/legal-bert-base-uncased",
    "bert-mini": "prajjwal1/bert-mini",
    "albert-base": "albert-base-v2",
}

id2label = {0: 'O', 1: 'B-RISK', 2: 'I-RISK'}
label2id = {'O': 0, 'B-RISK': 1, 'I-RISK': 2}

local_path = f"./gotcha-extractor-model/{model_choice}-optuna-search"
if not os.path.exists(local_path) or not os.path.exists(os.path.join(local_path, "config.json")):
    local_path = f"./gotcha-extractor-model/{model_choice}"
has_local = os.path.exists(local_path) and os.path.exists(os.path.join(local_path, "config.json"))

if has_local:
    model_path = local_path
    print(f"Loading locally trained model from: {model_path}")
else:
    model_path = fallback_map[model_choice]
    print(f"Local trained model not found. Falling back to base model: {model_path}")

tokenizer = AutoTokenizer.from_pretrained(model_path)
prev_verbosity = tf_logging.get_verbosity()
tf_logging.set_verbosity_error()
tf_logging.disable_progress_bar()
model = AutoModelForTokenClassification.from_pretrained(
    model_path,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)
tf_logging.enable_progress_bar()
tf_logging.set_verbosity(prev_verbosity)

model.eval()
print(f"Model loaded successfully!")


Loading locally trained model from: ./gotcha-extractor-model/electra-small
Model loaded successfully!


In [15]:
KEYWORDS_HIGH = [
    r"arbitrat", r"class\s+action", r"waiver", r"dispute",
    r"reserve\s+the\s+right\s+to", r"modify", r"revise", r"update", r"without\s+notice",
    r"sell", r"market", r"advertis", r"third\s+part",
    r"cannot\s+(ensure|warrant|guarantee)", r"no\s+warranty", r"indemni"
]

BOILERPLATE_PATTERNS = [
    r"this\s+privacy\s+policy\s+(\([^)]+\)\s+)?describes\s+the\s+practices",
    r"this\s+privacy\s+policy\s+applies\s+only\s+to",
    r"summary\s+the\s+notifications\s+provided\s+by\s+this\s+privacy\s+policy\s+include",
    r"^[a-zA-Z\s]+is\s+data\s+that\s+can\s+be\s+used\s+to\s+identify",
    r"^[a-zA-Z\s]+\s+means\s+any\s+information",
    r"legal\s+grounds\s+for\s+processing\s+personal\s+data",
    r"we\s+restrict\s+access\s+to\s+personal\s+information\s+collected.*to\s+our\s+employees",
    r"please\s+note\s+that\s+we\s+have\s+a\s+separate\s+privacy\s+disclosure\s+statement\s+to\s+address\s+our\s+protocols.*located\s+here",
    r"children\s+under\s+13", r"younger\s+than\s+13", r"receive\s+parental\s+consent",
    r"privacy\s+policy\s+effective\s+date"
]

def clean_boilerplate_header(sentence):
    sentence_clean = sentence.strip()
    sentence_lower = sentence_clean.lower()

    if re.match(r"^[A-Z\s\d/_:,,\'\"]{3,50}$", sentence_clean):
        return True

    for pattern in BOILERPLATE_PATTERNS:
        if re.search(pattern, sentence_lower):
            return True

    return False

def determine_risk_level(sentence, risk_tokens, has_high_keyword):
    if not risk_tokens:
        return None

    probs = [t["prob"] for t in risk_tokens]
    max_prob = max(probs)

    if max_prob >= 0.80 or (has_high_keyword and max_prob >= 0.68):
        return "HIGH RISK"
    elif has_high_keyword or max_prob >= 0.62:
        return "MEDIUM RISK"
    else:
        return "LOW RISK"

KEYWORDS_PRO_USER = [
    r"you\s+may\s+(access|correct|request\s+deletion|delete|port|object)",
    r"request\s+that\s+we\s+stop\s+(any\s+)?processing",
    r"freely\s+visit\s+our\s+(website|platform)\s+anonymously",
    r"without\s+being\s+required\s+to\s+provide\s+us\s+with\s+any\s+personal\s+information",
    r"rights\s+related\s+to\s+the\s+european\s+union",
    r"rights\s+related\s+to\s+gdpr",
    r"your\s+right\s+to\s+(access|delete|rectify|restrict)",
    r"opt[- ]out\s+of\s+receiving\s+(marketing|promotional|newsletter)",
    r"under\s+the\s+general\s+data\s+protection\s+regulation",
    r"right\s+to\s+request\s+that\s+we\s+disclose",
    r"right\s+to\s+know\s+what\s+personal\s+information",
]

def check_pro_user_override(sentence):
    sentence_clean = sentence.strip()
    sentence_lower = sentence_clean.lower()

    for pattern in KEYWORDS_PRO_USER:
        if re.search(pattern, sentence_lower):
            return True

    if re.search(r"\b(right(s)?\s+to|you\s+have\s+the\s+right\s+to)\s+.*\b(access|correct|delete|erase|rectify|update|portability|restrict)\b", sentence_lower):
        return True

    if re.search(r"\b(visit|browse)\b.*\banonymously\b", sentence_lower) and not re.search(r"\b(cannot|unable|restrict)\b", sentence_lower):
        return True

    if re.search(r"\brights\s+related\s+to\b.*\b(gdpr|ccpa|california\s+consumer|protection\s+regulation)\b", sentence_lower):
        return True

    return False


In [16]:

def extract_gotchas(raw_text, min_risk_tokens=3):
    if not raw_text or not raw_text.strip():
        return []

    cleaned_text = clean_text_pipeline(raw_text)
    sentences = sent_tokenize(cleaned_text)
    all_extracted = []

    for sentence in sentences:
        if not sentence.strip():
            continue

        if clean_boilerplate_header(sentence):
            continue

        if check_pro_user_override(sentence):
            continue

        inputs = tokenizer(
            sentence, 
            return_tensors="pt", 
            truncation=True, 
            max_length=512
        )
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        if isinstance(outputs, dict):
            logits = outputs['logits'][0]
        else:
            logits = outputs.logits[0]

        probs = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(logits, dim=-1)

        risk_tokens = []
        for t_idx, pred in enumerate(predictions):
            label = id2label[pred.item()]
            token_str = tokens[t_idx]
            if token_str in ('[CLS]', '[SEP]', '[PAD]'):
                continue
            prob = probs[t_idx][pred.item()].item()
            if label in ('B-RISK', 'I-RISK'):
                risk_tokens.append({"token": token_str, "prob": prob})

        if len(risk_tokens) >= min_risk_tokens:
            max_prob = max(t["prob"] for t in risk_tokens)

            has_high_keyword = False
            sentence_lower = sentence.lower()
            for pattern in KEYWORDS_HIGH:
                if re.search(pattern, sentence_lower):
                    has_high_keyword = True
                    break

            keep = False
            if has_high_keyword:
                if max_prob >= 0.55:
                    keep = True
            else:
                if max_prob >= 0.70:
                    keep = True

            if keep:
                level = determine_risk_level(sentence, risk_tokens, has_high_keyword)
                all_extracted.append((sentence, level))

    return all_extracted


In [17]:
sample_legalese = (
    "Welcome to the platform. By continuing, you agree to forced arbitration in the event of a dispute. "
    "We also reserve the right to sell your location data and usage habits to unverified third parties."
)

print("\n--- Test 1: Simple Inline Text ---")
print("Raw Text:", sample_legalese)
print("\nExtracted Gotchas:")
for idx, (clause, level) in enumerate(extract_gotchas(sample_legalese)):
    try:
        print(f"{idx + 1}. [{level}] {clause}")
    except UnicodeEncodeError:
        print(f"{idx + 1}. [{level}] {clause.encode('ascii', errors='replace').decode('ascii')}")



--- Test 1: Simple Inline Text ---
Raw Text: Welcome to the platform. By continuing, you agree to forced arbitration in the event of a dispute. We also reserve the right to sell your location data and usage habits to unverified third parties.

Extracted Gotchas:
1. [HIGH RISK] By continuing, you agree to forced arbitration in the event of a dispute.
2. [HIGH RISK] We also reserve the right to sell your location data and usage habits to unverified third parties.
